# embedding

In [8]:
import os
import json
import torch
import pickle
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
JSON_PATH = r'D:\Big_project_2025\Bao_Cao_TTTN\products_final.json'
TEXT_MODEL_PATH = r'D:\Big_project_2025\CODE_THTN\clip-ViT-B-32-multilingual-v1'
TEXT_EMBEDDING_FILE = "moho_text_embeddings.pkl"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 2. LOAD DỮ LIỆU VÀ MÔ HÌNH
# ==========================================
print("📂 Đang tải dữ liệu JSON...")
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    products_data = json.load(f)

print(f"🏗️ Đang khởi tạo mô hình Multilingual-CLIP từ: {TEXT_MODEL_PATH}")
model = SentenceTransformer(TEXT_MODEL_PATH, device=device)

# ==========================================
# 3. TRÍCH XUẤT TEXT EMBEDDINGS (CHIA THEO Ý)
# ==========================================
# Cấu trúc lưu trữ: { product_id: { 'name': vector, 'material': vector, 'description': vector } }
text_embeddings_dict = {}

print("🚀 Bắt đầu trích xuất Text Embeddings...")

for item in tqdm(products_data):
    p_id = item['product_id']
    
    # Lấy các trường thông tin
    name_text = item['content'].get('name', '')
    material_text = item['content'].get('material', '')
    desc_text = item['ai_metadata'].get('full_description', '')
    
    # Chuyển đổi từng trường thành vector (nếu trường đó không rỗng)
    # Chúng ta encode theo list để tận dụng batching bên trong model.encode
    texts_to_encode = [name_text, material_text, desc_text]
    
    # Encode (trả về list các numpy array)
    vectors = model.encode(texts_to_encode, show_progress_bar=False)
    
    # Lưu vào dictionary
    text_embeddings_dict[p_id] = {
        "name_vector": vectors[0],
        "material_vector": vectors[1],
        "description_vector": vectors[2],
        "metadata": {  # Lưu kèm text thô để tiện đối chiếu nếu cần
            "name": name_text,
            "material": material_text,
            "product_id": p_id
        }
    }

# ==========================================
# 4. LƯU KẾT QUẢ
# ==========================================
with open(TEXT_EMBEDDING_FILE, "wb") as f:
    pickle.dump(text_embeddings_dict, f)

print(f"✅ Đã trích xuất xong {len(text_embeddings_dict)} sản phẩm.")
print(f"💾 File lưu tại: {TEXT_EMBEDDING_FILE}")

# ==========================================
# 5. HÀM VÍ DỤ: CÁCH TÌM KIẾM TRÊN CÁC TRƯỜNG CHIA NHỎ
# ==========================================
def example_search(query_text, top_k=3):
    print(f"\n🔍 Thử nghiệm tìm kiếm: '{query_text}'")
    query_vector = model.encode([query_text], convert_to_tensor=True).to(device)
    
    results = []
    
    for p_id, vectors in text_embeddings_dict.items():
        # Chuyển numpy sang tensor
        v_name = torch.tensor(vectors['name_vector']).to(device)
        v_mat = torch.tensor(vectors['material_vector']).to(device)
        v_desc = torch.tensor(vectors['description_vector']).to(device)
        
        # Tính độ tương đồng với từng trường
        sim_name = torch.cosine_similarity(query_vector, v_name.unsqueeze(0)).item()
        sim_mat = torch.cosine_similarity(query_vector, v_mat.unsqueeze(0)).item()
        sim_desc = torch.cosine_similarity(query_vector, v_desc.unsqueeze(0)).item()
        
        # Lấy độ tương đồng cao nhất trong 3 trường (Max-Pooling)
        # Cách này giúp: nếu user tìm "gỗ sồi" thì trường material sẽ kéo score lên cao.
        best_score = max(sim_name, sim_mat, sim_desc)
        
        results.append({
            "product_id": p_id,
            "score": best_score,
            "name": vectors['metadata']['name']
        })
    
    # Sắp xếp và in kết quả
    results = sorted(results, key=lambda x: x['score'], reverse=True)[:top_k]
    for res in results:
        print(f"[{res['score']:.4f}] ID: {res['product_id']} - {res['name']}")

# Chạy thử nghiệm sau khi trích xuất
# example_search("bàn ăn gỗ cao su")
example_search("Bàn Ăn Gỗ Cao Su MOHO OSLO 901")

📂 Đang tải dữ liệu JSON...
🏗️ Đang khởi tạo mô hình Multilingual-CLIP từ: D:\Big_project_2025\CODE_THTN\clip-ViT-B-32-multilingual-v1


The tokenizer you are loading from 'D:\Big_project_2025\CODE_THTN\clip-ViT-B-32-multilingual-v1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


🚀 Bắt đầu trích xuất Text Embeddings...


100%|██████████| 193/193 [00:53<00:00,  3.61it/s]

✅ Đã trích xuất xong 193 sản phẩm.
💾 File lưu tại: moho_text_embeddings.pkl

🔍 Thử nghiệm tìm kiếm: 'Bàn Ăn Gỗ Cao Su MOHO OSLO 901'
[1.0000] ID: ban-an-oslo-901 - Bàn Ăn Gỗ Cao Su MOHO OSLO 901
[0.9815] ID: ghe-an-oslo-601 - Ghế Ăn Gỗ Cao Su Tự Nhiên MOHO OSLO 601
[0.9812] ID: ban-an-oslo-601 - Bộ Bàn Ăn Gỗ 4 Ghế Cao Su MOHO OSLO 601


# querry

In [13]:
import torch
import pickle
import os
from sentence_transformers import SentenceTransformer

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN (Phải khớp với file trích xuất)
# ==========================================
TEXT_EMBEDDING_FILE = "moho_text_embeddings.pkl"
TEXT_MODEL_PATH = r'D:\Big_project_2025\CODE_THTN\clip-ViT-B-32-multilingual-v1'
device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 2. NẠP DỮ LIỆU ĐÃ LƯU & MÔ HÌNH (Chạy 1 lần khi mở máy)
# ==========================================
print("📂 Đang nạp dữ liệu vector từ file .pkl...")
if not os.path.exists(TEXT_EMBEDDING_FILE):
    print(f"❌ Không tìm thấy file {TEXT_EMBEDDING_FILE}. Bạn cần chạy file trích xuất trước!")
else:
    with open(TEXT_EMBEDDING_FILE, "rb") as f:
        text_embeddings_dict = pickle.load(f)
    print(f"✅ Đã nạp {len(text_embeddings_dict)} sản phẩm.")

print(f"🏗️ Đang khởi tạo mô hình: {os.path.basename(TEXT_MODEL_PATH)}")
model = SentenceTransformer(TEXT_MODEL_PATH, device=device)

# ==========================================
# 3. HÀM TÌM KIẾM (Giữ nguyên logic chia 3 trường)
# ==========================================
def example_search(query_text, top_k=3):
    # 1. Chuyển câu hỏi của người dùng thành vector
    query_vector = model.encode([query_text], convert_to_tensor=True).to(device)
    
    results = []
    
    # 2. So sánh với từng sản phẩm trong database đã nạp từ file .pkl
    for p_id, data in text_embeddings_dict.items():
        # Chuyển numpy vector từ file pkl sang torch tensor
        v_name = torch.tensor(data['name_vector']).to(device)
        v_mat = torch.tensor(data['material_vector']).to(device)
        v_desc = torch.tensor(data['description_vector']).to(device)
        
        # Tính similarity cho từng trường
        # unsqueeze(0) để đưa vector về dạng batch size 1 (1, 512)
        sim_name = torch.cosine_similarity(query_vector, v_name.unsqueeze(0)).item()
        sim_mat = torch.cosine_similarity(query_vector, v_mat.unsqueeze(0)).item()
        sim_desc = torch.cosine_similarity(query_vector, v_desc.unsqueeze(0)).item()
        
        # Lấy score cao nhất (Max-pooling)
        best_score = max(sim_name, sim_mat, sim_desc)
        
        results.append({
            "product_id": p_id,
            "score": best_score,
            "name": data['metadata']['name'],
            "material": data['metadata']['material']
        })
    
    # 3. Sắp xếp kết quả từ cao xuống thấp
    results = sorted(results, key=lambda x: x['score'], reverse=True)[:top_k]
    
    print(f"\n🔍 Kết quả tìm kiếm cho: '{query_text}'")
    print("-" * 60)
    for res in results:
        print(f"[{res['score']:.4f}] ID: {res['product_id']}")
        print(f"           Tên: {res['name']}")
        print(f"           Chất liệu: {res['material']}")
        print("-" * 60)

📂 Đang nạp dữ liệu vector từ file .pkl...
✅ Đã nạp 193 sản phẩm.
🏗️ Đang khởi tạo mô hình: clip-ViT-B-32-multilingual-v1


The tokenizer you are loading from 'D:\Big_project_2025\CODE_THTN\clip-ViT-B-32-multilingual-v1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [19]:
# ==========================================
# 4. CHẠY TRUY VẤN LUÔN
# ==========================================
example_search("kệ tv")


🔍 Kết quả tìm kiếm cho: 'kệ tv'
------------------------------------------------------------
[0.9567] ID: ban-an-milan-901
           Tên: Bàn Ăn Gỗ MOHO MILAN 901 Màu Gỗ
           Chất liệu: Gỗ Tự Nhiên Câm Xe
------------------------------------------------------------
[0.9567] ID: ban-an-1
           Tên: Bàn Ăn Gỗ Tự Nhiên PLANK | Veneer Gỗ Sồi
           Chất liệu: Gỗ Tự Nhiên Câm Xe
------------------------------------------------------------
[0.9567] ID: ban-an-oslo-100
           Tên: BÀN ĂN TRÒN OSLO (MÀU NÂU 100)
           Chất liệu: Gỗ Tự Nhiên Câm Xe
------------------------------------------------------------


# xem kỉ hơn về vector đã en

In [21]:
import pickle

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
TEXT_EMBEDDING_FILE = r"D:\Big_project_2025\CODE_THTN\moho_text_embeddings.pkl"

# ==========================================
# ĐỌC FILE VÀ IN RA SẢN PHẨM ĐẦU TIÊN
# ==========================================
print("📂 Đang mở file pickle...")
with open(TEXT_EMBEDDING_FILE, "rb") as f:
    text_embeddings_dict = pickle.load(f)

# Lấy product_id đầu tiên (Key đầu tiên của dictionary)
first_product_id = list(text_embeddings_dict.keys())[0]
first_data = text_embeddings_dict[first_product_id]

print("\n==========================================")
print(f"🔑 Product ID (Key của Dictionary): {first_product_id}")
print("==========================================")

# 1. Xem thông tin văn bản gốc (Metadata)
print("\n📄 --- THÔNG TIN VĂN BẢN GỐC (Metadata) ---")
print(f"• Tên sản phẩm: {first_data['metadata']['name']}")
print(f"• Chất liệu:    {first_data['metadata']['material']}")
print(f"• Toàn bộ metadata được lưu:")
print(first_data['metadata'])


# 2. Xem cấu trúc của các Vector 512 chiều
print("\n🔢 --- CẤU TRÚC VECTOR (DỮ LIỆU SỐ HÓA) ---")

# Duyệt qua 3 loại vector mà bạn đã chia
for vec_name in ["name_vector", "material_vector", "description_vector"]:
    vector = first_data[vec_name]
    print(f"\n👉 Trường: {vec_name}")
    print(f"   - Kiểu dữ liệu: {type(vector)}")
    print(f"   - Số chiều (Shape): {vector.shape}") # Sẽ là (512,)
    print(f"   - 10 giá trị số thực đầu tiên: {vector[:10]}")  # In ra 10 giá trị đầu tiên để xem dạng số thực

📂 Đang mở file pickle...

🔑 Product ID (Key của Dictionary): ban-an-oslo-901

📄 --- THÔNG TIN VĂN BẢN GỐC (Metadata) ---
• Tên sản phẩm: Bàn Ăn Gỗ Cao Su MOHO OSLO 901
• Chất liệu:    Gỗ Công Nghiệp (MDF/MFC)
• Toàn bộ metadata được lưu:
{'name': 'Bàn Ăn Gỗ Cao Su MOHO OSLO 901', 'material': 'Gỗ Công Nghiệp (MDF/MFC)', 'product_id': 'ban-an-oslo-901'}

🔢 --- CẤU TRÚC VECTOR (DỮ LIỆU SỐ HÓA) ---

👉 Trường: name_vector
   - Kiểu dữ liệu: <class 'numpy.ndarray'>
   - Số chiều (Shape): (512,)
   - 10 giá trị số thực đầu tiên: [-0.05629741  0.00370236 -0.01931885 -0.10196123 -0.06048415 -0.07258573
 -0.08816873 -0.42410716 -0.06543051  0.07612041]

👉 Trường: material_vector
   - Kiểu dữ liệu: <class 'numpy.ndarray'>
   - Số chiều (Shape): (512,)
   - 10 giá trị số thực đầu tiên: [ 0.13755378 -0.19648457 -0.02300741  0.01255709  0.0375485  -0.02999129
 -0.16555801 -1.031125   -0.05144499  0.11990432]

👉 Trường: description_vector
   - Kiểu dữ liệu: <class 'numpy.ndarray'>
   - Số chiều (Shap

In [ ]:
# lưu một Dictionary (từ điển) của Python vào file. Cấu trúc phân cấp của nó như sau:
# {
#   "ID_Sản_Phẩm_1": {
#       "name_vector": [512 số],
#       "material_vector": [512 số],
#       "description_vector": [512 số],
#       "metadata": {"name": "...", "material": "..."}
#   },
#   "ID_Sản_Phẩm_2": {
#       ...
#   }
# }